# Stock Stars V2
Ranks the combined S&P 500, Nasdaq-100 and Dow 30 universe using only **returns + company strength**.

**No 13F / Whale / institutional ownership logic is used.**


In [ ]:
# 0. Install/update required packages (safe in Colab/Jupyter)
%pip -q install --upgrade yfinance pandas numpy requests lxml html5lib beautifulsoup4


In [ ]:
# 1. Imports and configuration
import io
import time
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import display

RETURNS_WEIGHT = 0.50
STRENGTH_WEIGHT = 0.50

RETURN_COMPONENT_WEIGHTS = {
    'ret_1y': 0.30,
    'ret_3y_ann': 0.25,
    'ret_5y_ann': 0.20,
    'relative_strength_1y': 0.10,
    'sharpe': 0.10,
    'risk_quality': 0.05,
}

STRENGTH_COMPONENT_WEIGHTS = {
    'roic': 0.25,
    'roe': 0.20,
    'earnings_growth': 0.20,
    'revenue_growth': 0.15,
    'profit_margin': 0.10,
    'financial_health': 0.10,
}


In [ ]:
# 2. Build the stock universe from S&P 500, Nasdaq-100 and Dow 30
HEADERS = {'User-Agent':'Mozilla/5.0 (StockStars research notebook)'}

def read_wiki_tables(url):
    r=requests.get(url,headers=HEADERS,timeout=30)
    r.raise_for_status()
    return pd.read_html(io.StringIO(r.text))

def sp500_members():
    tables=read_wiki_tables('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
    df=tables[0][['Symbol']].copy()
    df['Index']='S&P 500'
    return df

def nasdaq100_members():
    tables=read_wiki_tables('https://en.wikipedia.org/wiki/Nasdaq-100')
    for t in tables:
        symbol_col = 'Ticker' if 'Ticker' in t.columns else ('Symbol' if 'Symbol' in t.columns else None)
        if symbol_col and 90 <= len(t) <= 110:
            df=t[[symbol_col]].rename(columns={symbol_col:'Symbol'}).copy()
            df['Index']='Nasdaq-100'
            return df
    raise RuntimeError('Nasdaq-100 constituents table not found')

def dow30_members():
    tables=read_wiki_tables('https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average')
    for t in tables:
        symbol_col = 'Symbol' if 'Symbol' in t.columns else ('Ticker' if 'Ticker' in t.columns else None)
        if symbol_col and 25 <= len(t) <= 35:
            df=t[[symbol_col]].rename(columns={symbol_col:'Symbol'}).copy()
            df['Index']='Dow 30'
            return df
    raise RuntimeError('Dow 30 constituents table not found')

parts=[sp500_members(),nasdaq100_members(),dow30_members()]
membership=pd.concat(parts,ignore_index=True)
membership['Symbol']=membership['Symbol'].astype(str).str.upper().str.strip().str.replace('.','-',regex=False)
provenance=(membership.groupby('Symbol')['Index']
            .agg(lambda x: ', '.join(sorted(set(x))))
            .reset_index().rename(columns={'Symbol':'ticker','Index':'membership'}))
universe=provenance['ticker'].tolist()
print('Unique stocks:',len(universe))
display(provenance.head())


In [ ]:
# 3. Price-return metrics (also computes beta from market returns)
def annualized_return(series, years):
    s=series.dropna()
    if len(s)<2 or s.iloc[0] <= 0:
        return np.nan
    return (s.iloc[-1]/s.iloc[0])**(1/years)-1

spy_hist=yf.download('SPY',period='5y',auto_adjust=True,progress=False,threads=False)
if spy_hist.empty:
    raise RuntimeError('Could not download SPY benchmark data from Yahoo Finance')
spy_close=spy_hist['Close']
if isinstance(spy_close,pd.DataFrame):
    spy_close=spy_close.iloc[:,0]
spy_close=spy_close.dropna()
spy_returns=spy_close.pct_change().dropna()
spy_1y=(spy_close.tail(252).iloc[-1]/spy_close.tail(252).iloc[0]-1)

def price_metrics(ticker):
    try:
        hist=yf.download(ticker,period='5y',auto_adjust=True,progress=False,threads=False)
        if hist.empty or 'Close' not in hist:
            return {'ticker':ticker,'price_data_ok':False,'price_error':'no price history'}
        c=hist['Close']
        if isinstance(c,pd.DataFrame):
            c=c.iloc[:,0]
        c=c.dropna()
        if len(c)<200:
            return {'ticker':ticker,'price_data_ok':False,'price_error':'insufficient history'}
        r=c.pct_change().dropna()
        one_year=c.tail(min(252,len(c)))
        three_year=c.tail(min(756,len(c)))
        ret_1y=(one_year.iloc[-1]/one_year.iloc[0]-1) if len(one_year)>1 else np.nan
        ret_3y_ann=annualized_return(three_year,3) if len(three_year)>500 else np.nan
        ret_5y_ann=annualized_return(c,5) if len(c)>900 else np.nan
        avg_ann=r.mean()*252 if len(r)>20 else np.nan
        vol=r.std()*np.sqrt(252) if len(r)>20 else np.nan
        sharpe=avg_ann/vol if pd.notna(vol) and vol>0 else np.nan
        aligned=pd.concat([r.rename('stock'),spy_returns.rename('market')],axis=1).dropna()
        beta=(aligned['stock'].cov(aligned['market'])/aligned['market'].var()) if len(aligned)>30 and aligned['market'].var()>0 else np.nan
        return {'ticker':ticker,'price_data_ok':True,'price_error':'',
                'ret_1y':ret_1y,'ret_3y_ann':ret_3y_ann,'ret_5y_ann':ret_5y_ann,
                'avg_ann_return':avg_ann,'volatility':vol,'sharpe':sharpe,'beta':beta}
    except Exception as e:
        return {'ticker':ticker,'price_data_ok':False,'price_error':str(e)[:160]}


In [ ]:
# 4. Company-strength metrics from financial statements (no Yahoo .info calls)
def first_available(df,names):
    for name in names:
        if df is not None and not df.empty and name in df.index:
            s=pd.to_numeric(df.loc[name],errors='coerce').dropna()
            if not s.empty:
                return s
    return pd.Series(dtype=float)

def latest(series):
    return float(series.iloc[0]) if series is not None and len(series)>0 else np.nan

def yoy_growth(series):
    if series is None or len(series)<2:
        return np.nan
    current=float(series.iloc[0]); previous=float(series.iloc[1])
    if previous == 0 or not np.isfinite(previous):
        return np.nan
    return current/previous-1

def company_strength_metrics(ticker):
    try:
        stock=yf.Ticker(ticker)
        fin=stock.financials
        bs=stock.balance_sheet
        cf=stock.cashflow

        revenue=first_available(fin,['Total Revenue','Operating Revenue'])
        net_income=first_available(fin,['Net Income','Net Income Common Stockholders'])
        ebit=first_available(fin,['EBIT','Operating Income'])
        equity=first_available(bs,['Stockholders Equity','Total Stockholder Equity'])
        debt=first_available(bs,['Total Debt','Long Term Debt And Capital Lease Obligation','Long Term Debt'])
        operating_cf_s=first_available(cf,['Operating Cash Flow','Total Cash From Operating Activities'])
        capex=first_available(cf,['Capital Expenditure','Capital Expenditures'])

        ni=latest(net_income); rev=latest(revenue); eq=latest(equity); debt_now=latest(debt)
        roe=(ni/eq) if pd.notna(ni) and pd.notna(eq) and eq!=0 else np.nan
        profit_margin=(ni/rev) if pd.notna(ni) and pd.notna(rev) and rev!=0 else np.nan
        earnings_growth=yoy_growth(net_income)
        revenue_growth=yoy_growth(revenue)

        roic=np.nan
        if len(ebit)>0 and len(equity)>0:
            common=ebit.index.intersection(equity.index)
            if len(debt)>0:
                common=common.intersection(debt.index)
            if len(common)>0:
                invested=equity.loc[common] + (debt.loc[common] if len(debt)>0 else 0)
                nopat=ebit.loc[common]*0.79
                vals=(nopat/invested.replace(0,np.nan)).replace([np.inf,-np.inf],np.nan).dropna()
                if not vals.empty:
                    roic=float(vals.mean())

        operating_cf=latest(operating_cf_s)
        capex_now=latest(capex)
        free_cf=(operating_cf + capex_now) if pd.notna(operating_cf) and pd.notna(capex_now) else operating_cf
        debt_to_equity=(debt_now/eq) if pd.notna(debt_now) and pd.notna(eq) and eq!=0 else np.nan

        return {'ticker':ticker,'fundamental_data_ok':True,'fundamental_error':'',
                'roic':roic,'roe':roe,'profit_margin':profit_margin,
                'earnings_growth':earnings_growth,'revenue_growth':revenue_growth,
                'debt_to_equity':debt_to_equity,'operating_cashflow':operating_cf,
                'free_cashflow':free_cf}
    except Exception as e:
        return {'ticker':ticker,'fundamental_data_ok':False,'fundamental_error':str(e)[:160]}


In [ ]:
# 5. Collect metrics for the entire universe
price_rows=[]
strength_rows=[]
for i,t in enumerate(universe,1):
    price_rows.append(price_metrics(t))
    strength_rows.append(company_strength_metrics(t))
    if i % 25 == 0:
        print(f'Processed {i}/{len(universe)}')
    time.sleep(0.05)

price_df=pd.DataFrame(price_rows)
strength_df=pd.DataFrame(strength_rows)
df=(provenance.merge(price_df,on='ticker',how='left')
    .merge(strength_df,on='ticker',how='left'))
df['relative_strength_1y']=df['ret_1y']-spy_1y

print('Price failures:',int((~df['price_data_ok'].fillna(False)).sum()))
print('Fundamental failures:',int((~df['fundamental_data_ok'].fillna(False)).sum()))
display(df.head())


In [ ]:
# 6. Percentile scoring helpers
def pct_score(series,higher_is_better=True):
    x=pd.to_numeric(series,errors='coerce')
    s=x.rank(pct=True,method='average')*100
    return s if higher_is_better else 100-s

vol_score=pct_score(df['volatility'],higher_is_better=False)
beta_distance=(pd.to_numeric(df['beta'],errors='coerce')-1.0).abs()
beta_score=pct_score(beta_distance,higher_is_better=False)
df['risk_quality']=pd.concat([vol_score,beta_score],axis=1).mean(axis=1,skipna=True)

health_parts=pd.DataFrame({
    'ocf':df['operating_cashflow'].apply(lambda x:100 if pd.notna(x) and x>0 else (0 if pd.notna(x) else np.nan)),
    'fcf':df['free_cashflow'].apply(lambda x:100 if pd.notna(x) and x>0 else (0 if pd.notna(x) else np.nan)),
    'debt':pct_score(df['debt_to_equity'],higher_is_better=False),
})
df['financial_health']=health_parts.mean(axis=1,skipna=True)


In [ ]:
# 7. Returns Score (0-100)
return_scores=pd.DataFrame({
    'ret_1y':pct_score(df['ret_1y']),
    'ret_3y_ann':pct_score(df['ret_3y_ann']),
    'ret_5y_ann':pct_score(df['ret_5y_ann']),
    'relative_strength_1y':pct_score(df['relative_strength_1y']),
    'sharpe':pct_score(df['sharpe']),
    'risk_quality':df['risk_quality'],
})
df['returns_score']=sum(return_scores[k].fillna(50)*w for k,w in RETURN_COMPONENT_WEIGHTS.items())

# 8. Company Strength Score (0-100)
strength_scores=pd.DataFrame({
    'roic':pct_score(df['roic']),
    'roe':pct_score(df['roe']),
    'earnings_growth':pct_score(df['earnings_growth']),
    'revenue_growth':pct_score(df['revenue_growth']),
    'profit_margin':pct_score(df['profit_margin']),
    'financial_health':df['financial_health'],
})
df['strength_score']=sum(strength_scores[k].fillna(50)*w for k,w in STRENGTH_COMPONENT_WEIGHTS.items())

# Final score: equal emphasis on performance and company strength
df['final_score']=RETURNS_WEIGHT*df['returns_score'] + STRENGTH_WEIGHT*df['strength_score']


In [ ]:
# 9. Star assignment based on rank in the full universe
# Higher percentile = stronger overall stock.
df['final_percentile']=df['final_score'].rank(pct=True,method='average')*100

def stars_from_percentile(p):
    if pd.isna(p): return 'Unrated'
    if p>=90: return '★★★★★'
    if p>=75: return '★★★★'
    if p>=50: return '★★★'
    if p>=25: return '★★'
    return '★'

df['stars']=df['final_percentile'].apply(stars_from_percentile)
ranked=df.sort_values(['final_score','returns_score','strength_score'],ascending=False).reset_index(drop=True)
ranked.index += 1

cols=['ticker','membership','stars','final_score','returns_score','strength_score',
      'ret_1y','ret_3y_ann','ret_5y_ann','relative_strength_1y','sharpe',
      'roic','roe','earnings_growth','revenue_growth','profit_margin','financial_health']
display(ranked[cols].round(3))


In [ ]:
# 10. Selection lists and exports
five_star=ranked[ranked['stars'].eq('★★★★★')].copy()
four_or_five=ranked[ranked['stars'].isin(['★★★★★','★★★★'])].copy()

print('5-star stocks:',len(five_star))
print('4- or 5-star stocks:',len(four_or_five))
display(five_star[cols].round(3))

ranked.to_csv('stock_stars_ranked.csv',index_label='rank')
five_star.to_csv('stock_stars_5star.csv',index_label='rank')
four_or_five.to_csv('stock_stars_4plus.csv',index_label='rank')
print('Exports written.')
